# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa
## Proyecto Sello — Contaminación del Agua

Actividad individual (equivalente a la sección 4.1 de la guía S2), aplicada a un único archivo `lecturas_agua.csv` con **1,500,000 lecturas** de sensores de agua en 4 puntos: Río Coata, Juliaca (urbano), Cabanillas (ciudad) y Cabanillas (nacimiento de agua).

Cubre los 5 parámetros de calidad de agua del proyecto: **pH**, **turbidez**, **metales pesados** (plomo, arsénico, mercurio, cadmio), **bacterias y parásitos** (coliformes fecales, *E. coli*, presencia de parásitos) y **radiactividad**.

## 1. SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("proyecto-agua-fundamentos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/28 02:59:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"

## 2. Cargar y explorar (primera lectura: `inferSchema`)

`sensor_id` es una cadena de solo dígitos (`"0001"`, `"0002"`, ...) — con `inferSchema=True` Spark puede leerla como número y perder el cero inicial (el mismo riesgo que `article_id` en la guía del curso).

In [4]:
df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    inferSchema=True,
)

df_agua.printSchema()
df_agua.show(5, truncate=False)

root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: integer (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nulla

Verifica si `sensor_id` perdió el cero inicial. Corrígelo con un esquema explícito:

In [5]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

schema_agua = StructType([
    StructField("id_lectura", IntegerType(), True),
    StructField("sensor_id", StringType(), True),        # <- texto, conserva el cero inicial
    StructField("ubicacion", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
    StructField("canal_transmision_id", IntegerType(), True),
    StructField("ph", DoubleType(), True),
    StructField("turbidez_ntu", DoubleType(), True),
    StructField("temperatura_c", DoubleType(), True),
    StructField("conductividad_us_cm", DoubleType(), True),
    StructField("solidos_disueltos_totales_mg_l", DoubleType(), True),
    StructField("oxigeno_disuelto_mg_l", DoubleType(), True),
    StructField("plomo_mg_l", DoubleType(), True),
    StructField("arsenico_mg_l", DoubleType(), True),
    StructField("mercurio_mg_l", DoubleType(), True),
    StructField("cadmio_mg_l", DoubleType(), True),
    StructField("coliformes_fecales_nmp_100ml", DoubleType(), True),
    StructField("escherichia_coli_nmp_100ml", DoubleType(), True),
    StructField("presencia_parasitos", IntegerType(), True),
    StructField("radiactividad_bq_l", DoubleType(), True),
    StructField("caudal_l_s", DoubleType(), True),
    StructField("indice_riesgo_normalizado", DoubleType(), True),
])

df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    schema=schema_agua,
)

df_agua.printSchema()

root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nullab

In [6]:
num_filas, num_cols = df_agua.count(), len(df_agua.columns)
print(f"Filas: {num_filas}, Columnas: {num_cols}")

df_agua.select(
    "ph", "turbidez_ntu", "plomo_mg_l", "radiactividad_bq_l", "escherichia_coli_nmp_100ml"
).describe().show()

Filas: 1500000, Columnas: 21


26/08/28 03:00:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:===>                                                     (1 + 15) / 16]

+-------+------------------+------------------+--------------------+-------------------+--------------------------+
|summary|                ph|      turbidez_ntu|          plomo_mg_l| radiactividad_bq_l|escherichia_coli_nmp_100ml|
+-------+------------------+------------------+--------------------+-------------------+--------------------------+
|  count|           1500000|           1500000|             1500000|            1500000|                   1500000|
|   mean| 6.779985460000028|23.894756653333424|0.030705033733334528| 0.3199175309999987|         393.7583446666667|
| stddev|0.5663791440182202| 21.34430490748802| 0.04163392565905887|0.39692497518627917|        475.04581866986206|
|    min|              3.98|               0.0|                 0.0|                0.0|                       0.0|
|    max|              8.87|            136.68|               0.325|              2.701|                    4035.0|
+-------+------------------+------------------+--------------------+----

## 3. Transformaciones y evaluación perezosa

`select()` y `filter()` encadenados no ejecutan nada todavía — solo construyen el plan.

In [7]:
from pyspark.sql.functions import col

df_transformado = (
    df_agua
    .select("ubicacion", "sensor_id", "fecha_hora", "ph", "plomo_mg_l", "radiactividad_bq_l")
    .filter(col("ubicacion") == "Rio Coata")
)

df_transformado

DataFrame[ubicacion: string, sensor_id: string, fecha_hora: timestamp, ph: double, plomo_mg_l: double, radiactividad_bq_l: double]

In [8]:
df_transformado.show(10, truncate=False)
df_transformado.count()

+---------+---------+-------------------+----+----------+------------------+
|ubicacion|sensor_id|fecha_hora         |ph  |plomo_mg_l|radiactividad_bq_l|
+---------+---------+-------------------+----+----------+------------------+
|Rio Coata|0004     |2024-07-02 23:00:00|6.18|0.0307    |0.3034            |
|Rio Coata|0035     |2025-06-26 14:00:00|6.45|0.0124    |1.0221            |
|Rio Coata|0059     |2025-07-14 03:00:00|5.74|0.0785    |0.2556            |
|Rio Coata|0002     |2025-04-26 00:00:00|5.72|0.0737    |0.6065            |
|Rio Coata|0055     |2025-08-13 17:00:00|6.3 |0.0687    |0.4677            |
|Rio Coata|0023     |2025-06-21 07:00:00|5.04|0.0       |0.8183            |
|Rio Coata|0057     |2024-09-17 14:00:00|6.62|0.0432    |0.8593            |
|Rio Coata|0075     |2024-05-29 22:00:00|6.32|0.1511    |0.7038            |
|Rio Coata|0058     |2024-09-02 00:00:00|6.16|0.0781    |1.034             |
|Rio Coata|0063     |2024-12-01 09:00:00|5.8 |0.1183    |0.4359            |

420023

## 4. Plan de ejecución con `explain()`

In [9]:
df_transformado.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('ubicacion, Rio Coata)
+- Project [ubicacion#126, sensor_id#125, fecha_hora#127, ph#129, plomo_mg_l#135, radiactividad_bq_l#142]
   +- Relation [id_lectura#124,sensor_id#125,ubicacion#126,fecha_hora#127,canal_transmision_id#128,ph#129,turbidez_ntu#130,temperatura_c#131,conductividad_us_cm#132,solidos_disueltos_totales_mg_l#133,oxigeno_disuelto_mg_l#134,plomo_mg_l#135,arsenico_mg_l#136,mercurio_mg_l#137,cadmio_mg_l#138,coliformes_fecales_nmp_100ml#139,escherichia_coli_nmp_100ml#140,presencia_parasitos#141,radiactividad_bq_l#142,caudal_l_s#143,indice_riesgo_normalizado#144] csv

== Analyzed Logical Plan ==
ubicacion: string, sensor_id: string, fecha_hora: timestamp, ph: double, plomo_mg_l: double, radiactividad_bq_l: double
Filter (ubicacion#126 = Rio Coata)
+- Project [ubicacion#126, sensor_id#125, fecha_hora#127, ph#129, plomo_mg_l#135, radiactividad_bq_l#142]
   +- Relation [id_lectura#124,sensor_id#125,ubicacion#126,fecha_hora#127,canal_transmis

## 5. Funciones con `withColumn()` — clasificar por los 5 parámetros

Umbrales de referencia: pH aceptable 6.5-8.5; plomo > 0.01 mg/L y arsénico > 0.01 mg/L se consideran altos (referencia OMS); presencia de parásitos es cualquier detección positiva; radiactividad > 0.5 Bq/L se marca como alta en este proyecto.

In [10]:
from pyspark.sql.functions import col, when, lit, current_date

df_agua = df_agua.withColumn(
    "ph_dentro_rango_potable",
    when((col("ph") >= 6.5) & (col("ph") <= 8.5), "Si").otherwise("No")
)

df_agua = df_agua.withColumn(
    "riesgo_metales",
    when((col("plomo_mg_l") > 0.01) | (col("arsenico_mg_l") > 0.01), "Alto")
    .when((col("plomo_mg_l") > 0.005) | (col("arsenico_mg_l") > 0.005), "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn(
    "riesgo_biologico",
    when((col("presencia_parasitos") == 1) | (col("escherichia_coli_nmp_100ml") > 500), "Alto")
    .when(col("coliformes_fecales_nmp_100ml") > 500, "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn(
    "nivel_radiactividad",
    when(col("radiactividad_bq_l") > 0.5, "Alto")
    .when(col("radiactividad_bq_l") > 0.2, "Moderado")
    .otherwise("Bajo")
)

df_agua = df_agua.withColumn("fuente_dato", lit("Sensor IoT - Proyecto Sello"))
df_agua = df_agua.withColumn("fecha_procesado", current_date())

df_agua.select(
    "ubicacion", "ph", "ph_dentro_rango_potable", "riesgo_metales",
    "riesgo_biologico", "nivel_radiactividad"
).show(10, truncate=False)

+-------------------------------+----+-----------------------+--------------+----------------+-------------------+
|ubicacion                      |ph  |ph_dentro_rango_potable|riesgo_metales|riesgo_biologico|nivel_radiactividad|
+-------------------------------+----+-----------------------+--------------+----------------+-------------------+
|Rio Coata                      |6.18|No                     |Alto          |Alto            |Moderado           |
|Cabanillas (nacimiento de agua)|7.38|Si                     |Bajo          |Bajo            |Bajo               |
|Juliaca (urbano)               |7.07|Si                     |Alto          |Moderado        |Moderado           |
|Rio Coata                      |6.45|No                     |Alto          |Moderado        |Alto               |
|Juliaca (urbano)               |7.09|Si                     |Alto          |Alto            |Moderado           |
|Juliaca (urbano)               |7.03|Si                     |Alto          |Alt

## 6. Agrupaciones y agregaciones

In [11]:
from pyspark.sql.functions import avg, count, max as spark_max

resumen_por_ubicacion = df_agua.groupBy("ubicacion").agg(
    count("*").alias("num_lecturas"),
    avg("ph").alias("ph_promedio"),
    avg("plomo_mg_l").alias("plomo_promedio"),
    spark_max("plomo_mg_l").alias("plomo_maximo"),
    avg("escherichia_coli_nmp_100ml").alias("ecoli_promedio"),
    avg("presencia_parasitos").alias("proporcion_con_parasitos"),
    avg("radiactividad_bq_l").alias("radiactividad_promedio"),
)

resumen_por_ubicacion.orderBy(col("radiactividad_promedio").desc()).show(truncate=False)

[Stage 14:===>                                                    (1 + 15) / 16]

+-------------------------------+------------+------------------+---------------------+------------+------------------+------------------------+----------------------+
|ubicacion                      |num_lecturas|ph_promedio       |plomo_promedio       |plomo_maximo|ecoli_promedio    |proporcion_con_parasitos|radiactividad_promedio|
+-------------------------------+------------+------------------+---------------------+------------+------------------+------------------------+----------------------+
|Rio Coata                      |420023      |6.2004417853308045|0.0811331127104945   |0.325       |354.7994538394326 |0.11970534946895765     |0.852332138716213     |
|Juliaca (urbano)               |420063      |6.799377093435995 |0.020671474040798617 |0.089       |917.3304909025551 |0.17961591475564379     |0.15112300678707738   |
|Cabanillas (ciudad)            |360166      |7.000391125203382 |0.00825225812541998  |0.0368      |154.22067046861724|0.06003620552745123     |0.12044984451613

In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg as avg_

ventana_ubicacion = Window.partitionBy("ubicacion")

df_con_promedio = df_agua.withColumn(
    "ph_promedio_ubicacion",
    avg_("ph").over(ventana_ubicacion)
)

df_con_promedio.select("ubicacion", "sensor_id", "ph", "ph_promedio_ubicacion").show(10, truncate=False)

[Stage 19:>                                                         (0 + 1) / 1]

+-------------------------------+---------+----+---------------------+
|ubicacion                      |sensor_id|ph  |ph_promedio_ubicacion|
+-------------------------------+---------+----+---------------------+
|Cabanillas (nacimiento de agua)|0277     |7.38|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0247     |7.26|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0264     |7.64|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0251     |7.21|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0304     |6.95|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0314     |7.38|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0249     |7.43|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0253     |7.25|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0303     |7.31|7.300066789436449    |
|Cabanillas (nacimiento de agua)|0275     |7.63|7.300066789436449    |
+-------------------------------+---------+----+---------------------+
only s

## 7. RDD — `map`, `flatMap`, `filter`, `reduceByKey`

Por cada lectura, identifica qué parámetros salieron fuera de rango (puede ser ninguno, uno o varios) y cuenta, sobre las 1,500,000 lecturas, cuántas veces se excedió cada parámetro en total — un solo `reduceByKey` sobre todo el dataset.

In [13]:
def problemas_de_la_lectura(fila):
    problemas = []
    if fila.ph < 6.5 or fila.ph > 8.5:
        problemas.append("ph_fuera_de_rango")
    if fila.plomo_mg_l > 0.01:
        problemas.append("plomo_alto")
    if fila.arsenico_mg_l > 0.01:
        problemas.append("arsenico_alto")
    if fila.mercurio_mg_l > 0.001:
        problemas.append("mercurio_alto")
    if fila.presencia_parasitos == 1:
        problemas.append("parasitos_presentes")
    if fila.radiactividad_bq_l > 0.5:
        problemas.append("radiactividad_alta")
    return problemas

rdd_lecturas = df_agua.select(
    "ph", "plomo_mg_l", "arsenico_mg_l", "mercurio_mg_l", "presencia_parasitos", "radiactividad_bq_l"
).rdd

# flatMap: cada lectura puede producir 0, 1 o varios "problemas" -> se aplanan en un solo RDD
rdd_problemas = rdd_lecturas.flatMap(problemas_de_la_lectura)

# filter: descarta cualquier valor vacio (no deberia haber, pero es la practica pedida)
rdd_problemas = rdd_problemas.filter(lambda p: p != "")

# map: convierte cada problema en un par (problema, 1)
pares = rdd_problemas.map(lambda p: (p, 1))

# reduceByKey: suma cuantas veces aparecio cada tipo de problema en todo el dataset
from operator import add
conteo_problemas = pares.reduceByKey(add)

conteo_problemas.takeOrdered(10, key=lambda x: -x[1])

[('plomo_alto', 832083),
 ('arsenico_alto', 740088),
 ('mercurio_alto', 637587),
 ('ph_fuera_de_rango', 434447),
 ('radiactividad_alta', 339664),
 ('parasitos_presentes', 150342)]

In [14]:
rdd_riesgo = df_agua.select("ubicacion", "riesgo_biologico").rdd

pares_riesgo_alto = (
    rdd_riesgo
    .filter(lambda fila: fila.riesgo_biologico == "Alto")
    .map(lambda fila: (fila.ubicacion, 1))
)

conteo_riesgo_alto = pares_riesgo_alto.reduceByKey(add)
conteo_riesgo_alto.collect()

[('Cabanillas (ciudad)', 21870),
 ('Juliaca (urbano)', 333075),
 ('Cabanillas (nacimiento de agua)', 2990),
 ('Rio Coata', 141194)]